# 01 · السحب
هذا النوتبوك للاستكشاف والتأكد. التشغيل التلقائي كل ساعة يستخدم **نفس الدوال** من مجلد `pipeline/` عن طريق `python -m pipeline.run`.

استخدمه لما:
- تبي تشوف شكل البيانات الخام
- يتغيّر موقع دار وينكسر السحب، وتبي تعرف وين
- تبي تجرّب دار جديدة قبل ما تفعّلها

In [ ]:
import sys, logging
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
import pandas as pd
pd.set_option("display.max_rows", 60)

In [ ]:
from pipeline.config import load_config
from pipeline.http import PoliteSession
from pipeline.scrapers import vox

cfg = load_config()
st = cfg["settings"]
session = PoliteSession(st["user_agent"], delay=st["request_delay_seconds"])
cfg["chains"]["vox"]["mode"], st["cities"]

## 1) هل robots.txt يسمح؟
لازم يطلع `True` للصفحات اللي نسحبها و `False` للممنوعة.

In [ ]:
for url in ["https://ksa.voxcinemas.com/showtimes/riyadh-park-riyadh",
            "https://ksa.voxcinemas.com/showtimes?c=riyadh-park-riyadh&d=20261008",
            "https://ksa.voxcinemas.com/booking/0034-225818"]:
    print(session.allowed(url), url)

## 2) قائمة الفروع من خريطة الموقع

In [ ]:
from pipeline.scrapers import city_from_slug
branches = vox.list_branches(session, cfg["chains"]["vox"]["sitemap"])
pd.DataFrame({"slug": branches, "city": [city_from_slug(b) for b in branches]})

## 3) جرّب فرع واحد
لو طلع صفر عروض، غالباً تغيّر شكل الصفحة. افتح `html[:3000]` وشوف وين صارت روابط `/booking/`.

In [ ]:
slug = "riyadh-park-riyadh"
url = f"https://ksa.voxcinemas.com/showtimes/{slug}"
html = session.get(url).text
shows = vox.parse_showtimes_html(html, slug, url)
raw_one = pd.DataFrame([s.to_dict() for s in shows])
print(len(raw_one), "عرض")
raw_one.head(20)

In [ ]:
# تشخيص سريع: كم رابط حجز في الصفحة؟ لو فيه روابط والمحلّل رجّع صفر، المشكلة في ربطها بالفلم
from bs4 import BeautifulSoup
soup = BeautifulSoup(html, "lxml")
links = soup.select('a[href*="/booking/"]')
print(len(links), "رابط حجز")
[(a.get_text(strip=True), a["href"]) for a in links[:10]]

## 4) كل الفروع في المدن المختارة
يأخذ وقت لأننا ننتظر بين كل طلب وطلب.

In [ ]:
all_shows = vox.scrape(session, cfg["chains"]["vox"], st.get("cities"))
raw = pd.DataFrame([s.to_dict() for s in all_shows])
raw.to_pickle(ROOT / "data" / "raw_latest.pkl")
raw.groupby("cinema_slug").size().sort_values(ascending=False)